# Работа с LLM через API: GigaChat + LangChain

Практическое занятие 2. Курс «Введение в профессию».

Задача: автоматически извлекать из текстовых заявок на аренду жилья количество проживающих, используя модель GigaChat, подключённую через `langchain-gigachat`.

## 1. Импортируем компоненты LangChain

- `GigaChat` — интерфейс модели;
- `SystemMessagePromptTemplate`, `HumanMessagePromptTemplate` — шаблоны системного и пользовательского сообщений;
- `PromptTemplate`, `ChatPromptTemplate` — сборка промптов;
- `StrOutputParser` — парсинг ответа как строки;
- `os` и `dotenv` — безопасное чтение ключа из `.env`;
- `pandas` — работа с таблицей заявок.

In [ ]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

# Ключ читается из .env (переменная GIGA_KEY), а не хранится в коде
load_dotenv()
api_key = os.getenv("GIGA_KEY")
if not api_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

## 2. Инициализация модели и проверка подключения

In [ ]:
llm = GigaChat(
    credentials=api_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

response = llm.invoke("Привет! Как дела?")
print(response.content)

## 3. Системное и пользовательское промптирование

Системное сообщение задаёт роль модели, пользовательское — конкретную заявку и инструкцию.

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья."
)
user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\nИзвлеки количество человек и верни только число."
)
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

chain = chat_prompt | llm | StrOutputParser()
print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))

## 4. Базовый промпт

Простой `PromptTemplate` + `StrOutputParser`: модель возвращает только число.

In [ ]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain_basic = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]
for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain_basic.invoke({'text': text})}")
    print("---")

## 5. Загружаем 15 заявок варианта 19

Файл `rental_19.csv` содержит столбец `text` (заявка) и `amount` (эталонное количество проживающих).

In [ ]:
df = pd.read_csv("rental_19.csv", sep=";", encoding="utf-8")
df.head()

## 6. Прогоняем каждую заявку через модель

Идём по столбцу `text`, собираем ответы и записываем их в новый столбец `result`, затем сохраняем таблицу в CSV.

In [ ]:
results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain_basic.invoke({"text": text})
    except Exception as e:  # noqa: BLE001
        result = f"ERROR: {e}"
    results.append(result)

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["amount", "result"]]